# Estimation and Prediction


## OLS

Let's have a look at the model structure in the nwcst program.

```
ols_fit, ols_vars = fit_ols( 
    yt , 
    xt ,
    )
```

This is a function (```fit_ols```) that takes the y variable and x variable, and returns the model fit (ols_fit) and the variables used (ols_vars).

When we run ```fit_ols```, the program calls the following:

```
def fit_ols(
    ytrain ,
    xtrain ,
    target_variable = target_variable,
    ) :
    
    model = LinearRegression()
    return model.fit(xtrain, ytrain) , xtrain.columns
```

In this function, the actual modelling is done in 

```
model = LinearRegression()
model.fit(xtrain, ytrain) 
```

In the program, predicted values are estimated and saved as follows:

```
def predict_ols(
    model ,
    X ,
    train_vars ,
    date ,
    pred_dict ,
    l ,
    target_variable = target_variable ,
    ) :
    X = X[train_vars]
    
    pred = model.predict(X)[0]
    pred_dict[l].append(pred)
    return pred_dict
```

But this is simply a function that organizes the predict function in the model:

```
pred = model.predict(X)
pred
```

To learn how to run models, we can simplify this process in a simple database.

## Data

In [ ]:
import os
import pandas as pd
import numpy as np
PATH_DATA = f"{os.getcwd()}/../data"

data = pd.read_csv(f"{PATH_DATA}/data.csv", parse_dates=["date"], index_col='date')
metadata = pd.read_csv(f"{PATH_DATA}/meta_full.csv")

meta_col = 'ticket'
cpi = 'MCPI0000'
# this loops deflates nominal variables using the cpi
base = data[cpi][data.index.year == 2018].mean()
nominal = metadata.loc[metadata['nominal']==1][meta_col].values
for column in data.columns :
    if column in nominal : data[column] = data[column]*data[cpi]/base

from statsmodels.tsa.seasonal import seasonal_decompose


# this loop seasonally-adjusts the variables
quarterly = metadata.loc[metadata['freq']=='q'][meta_col].values
monthly = metadata.loc[metadata['freq']=='m'][meta_col].values
weekly = metadata.loc[metadata['freq']=='w'][meta_col].values
daily = metadata.loc[metadata['freq']=='d'][meta_col].values
for column in data.columns :
    if len(data[column].dropna(how='all', axis=0))>12 :
        if column in quarterly :
            data[column] = seasonal_decompose(data[column].dropna(how='all', axis=0), model='additive', extrapolate_trend='freq', period=4).trend
        elif column in monthly :
            data[column] = seasonal_decompose(data[column].dropna(how='all', axis=0), model='additive', extrapolate_trend='freq', period=12).trend
        elif column in daily :
            data[column] = seasonal_decompose(data[column].dropna(how='all', axis=0), model='additive', extrapolate_trend='freq', period=31).trend

# this loop calculates growth rates
for column in data.columns :
    if column in quarterly : data[column] = data[column].dropna(how='all', axis=0).pct_change()*100
    else : data[column] = data[column].dropna(how='all', axis=0).pct_change()*100


data.drop(cpi, axis=1, inplace=True)
data.replace([np.inf, -np.inf], np.nan, inplace=True)
data.dropna(how='all', axis=1, inplace=True)

data = data[~(data.index < '2015-01-01')].reset_index(drop=False)
data = data.set_index('date').dropna(how='all', axis=0)

data = data[['RGDP0000', 'UGTR1001', "UBMB0000", 'RARR0000', 'EBRE0000', 'XSP50000', 'XIMP0000']]

agg_map = metadata.set_index('ticket')['agg_method'].to_dict()
agg_funcs = {
    col: (agg_map[col] if agg_map[col] in ['mean', 'last', 'sum'] else 'mean')
    for col in data.columns if col in agg_map
}
data = data.resample('QS').agg(agg_funcs)
data = data[data.index>"2015-01-01"]
data.dropna(how='any', axis=0, inplace=True)
data.tail(5)

,RGDP0000,UGTR1001,UBMB0000,RARR0000,EBRE0000,XSP50000,XIMP0000
date,,,,,,,
2024-04-01,2.611616,0.045910,0.988946,3.495760,-1.312720,2.162853,-6.101173
2024-07-01,2.568534,0.236662,0.754458,2.666507,-0.819028,1.342295,-6.275980
2024-10-01,1.168062,-0.099903,1.131789,2.859919,-1.843689,0.690465,-2.682353
2025-01-01,3.422686,0.051047,0.743561,1.400856,-1.269869,1.252991,28.181088
2025-04-01,2.256652,0.370587,0.136195,3.003035,-1.433802,1.044989,-15.214367


With the dataset above, we can do a simple GDP estimation with OLS as follows:

In [13]:
train = data[data.index < '2024-01-01']
test = data[data.index >= '2024-01-01']

X_train = train.drop('RGDP0000', axis=1)
y_train = train['RGDP0000']

X_test = test.drop('RGDP0000', axis=1)
y_obs = test['RGDP0000']

## OLS Estimation

In [15]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train) 

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


An prediction:

In [17]:
pred = model.predict(X_test)
pred

array([-0.22218339,  1.32118529,  1.3851533 ,  0.04723151, -1.69204407,
        1.76320425])

You can now compare with observed values

In [25]:
Table = pd.DataFrame({'Observed': y_obs, 'OLS predicted': pred}, index=y_obs.index)
# Estimate how far off in percent terms is the predicted value from observed
Table['Percent Error'] = (Table['OLS predicted']/Table['Observed']) - 1
Table

,Observed,OLS predicted,Percent Error
date,,,
2024-01-01,2.213659,-0.222183,-1.100369
2024-04-01,2.611616,1.321185,-0.494112
2024-07-01,2.568534,1.385153,-0.460722
2024-10-01,1.168062,0.047232,-0.959564
2025-01-01,3.422686,-1.692044,-1.494361
2025-04-01,2.256652,1.763204,-0.218663


And estimate RMSE

In [23]:
np.sqrt(np.mean((np.array(Table['Observed']) - np.array(Table['OLS predicted'])) ** 2))

np.float64(2.4718127628436344)

## Machine Learning models


Use the sklearn package to estimate OLS Ridge, LASSO, Elasticnet, Random Forest, Decision Tree, GBT, and LSTM.

- Estimate the model in the train data
- Predict the outcome in the test data
- Compare predicted and observed values
- Compute the RMSE


**Hints:** i) Be sure to load the packages; ii) You may want to use inspiration from our helper functions.


### Example


```
'''
DEFINE ALL OLS FUNCTIONS :
    fit: model training
    predict: test fit
    oos: out of sample prediction
'''
def fit_ols(
    ytrain ,
    xtrain ,
    target_variable = target_variable,
    ) :
    
    model = LinearRegression()
    return model.fit(xtrain, ytrain) , xtrain.columns

def predict_ols(
    model ,
    X ,
    train_vars ,
    date ,
    pred_dict ,
    l ,
    target_variable = target_variable ,
    ) :
    X = X[train_vars]
    
    pred = model.predict(X)[0]
    pred_dict[l].append(pred)
    return pred_dict

'''
DEFINE ALL OLS RIDGE FUNCTIONS
'''

def fit_olsr(
    ytrain ,
    xtrain ,
    target_variable = target_variable,
    alphas = [0.0001, 0.001, 0.01, 0.1, 1, 10, 20],
    ) :
    
    model = RidgeCV( alphas = alphas )
    return model.fit(xtrain, ytrain) , xtrain.columns

def predict_olsr(
    model ,
    X ,
    train_vars ,
    date ,
    pred_dict ,
    l ,
    target_variable = target_variable,
    ) :
    X = X[train_vars]
    
    pred = model.predict(X)[0]
    pred_dict[l].append(pred)
    return pred_dict

'''
DEFINE ALL ENET FUNCTIONS
'''
def fit_enet(
    ytrain,
    xtrain ,
    target_variable = target_variable,
    params = {
        'alpha' : 1e-5 ,
        'l1_ratio' : 0.25 ,
    }
    ) :

    model = ElasticNet(alpha = params['alpha'] , l1_ratio = params['l1_ratio'] )
    return model.fit(xtrain, ytrain) , xtrain.columns

def predict_enet(
    model ,
    X ,
    train_vars ,
    date ,
    pred_dict ,
    l ,
    target_variable = target_variable,
    ) :
    X = X[train_vars]
    
    pred = model.predict(X)[0]
    pred_dict[l].append(pred)
    return pred_dict


'''
DEFINE LASSO FUNCTIONS
'''

def fit_lasso(
    ytrain, 
    xtrain ,
    target_variable = target_variable,
    alpha = 1e-5,
    ) :
    
    model = Lasso( alpha = alpha )
    
    return model.fit(xtrain, ytrain) , xtrain.columns

def predict_lasso(
    model ,
    X ,
    train_vars ,
    date ,
    pred_dict ,
    l ,
    target_variable = target_variable,
    ) :
    X = X[train_vars]
    
    pred = model.predict(X)[0]
    pred_dict[l].append(pred)
    return pred_dict

'''
DEFINE DT FUNCTIONS
'''

def fit_dt(
    ytrain ,
    xtrain ,
    target_variable = target_variable,
    ModelN = 200 ,
    ) :
    
    models = []
    for i in range(ModelN):
        model = DecisionTreeRegressor(criterion = "absolute_error", 
                                      min_samples_split = 6, 
                                      min_samples_leaf = 2)
        
        model.fit(xtrain, ytrain)
        models.append(model)
    
    return models , xtrain.columns

def predict_dt(
    model ,
    X ,
    train_vars ,
    date ,
    pred_dict ,
    l ,
    target_variable = target_variable,
    ) :
    X = X[train_vars]
    
    preds = []
    for mod in model:
        prediction = mod.predict(X)[0]
        preds.append(prediction)
    
    pred_dict[l].append(np.nanmean(preds))
    return pred_dict


'''
DEFINE RF FUNCTIONS
'''

def fit_rf(
    ytrain ,
    xtrain , 
    target_variable = target_variable,
    ModelN = 200 , 
    n_estimators = 130 ,
    ) :

    models = []
    for i in range(ModelN):
        model = RandomForestRegressor(
            n_estimators=n_estimators,  
            criterion = "absolute_error", 
            max_features=18, 
            min_samples_split=4, 
            min_samples_leaf=2
            )
        
        model.fit(xtrain, ytrain)
        models.append(model)
    
    return models , xtrain.columns

def predict_rf(
    model ,
    X ,
    train_vars ,
    date ,
    pred_dict ,
    l ,
    target_variable = target_variable,
    ) :
    X = X[train_vars]
    
    preds = []
    for mod in model:
        prediction = mod.predict(X)[0]
        preds.append(prediction)
    
    pred_dict[l].append(np.nanmean(preds))
    return pred_dict

'''
DEFINE GBT FUNCTIONS
'''

def fit_gbt(
    ytrain, 
    xtrain ,
    target_variable = target_variable,
    ModelN = 200 , 
    n_estimators = 100 ,
    learning = 0.15 ,
    ) :
        
    models = []
    for i in range(ModelN):
        model = GradientBoostingRegressor(
                    n_estimators=100, 
                    learning_rate=learning, 
                    loss='absolute_error', 
                    min_samples_split=6, 
                    min_samples_leaf=3
                )
        
        model.fit(xtrain, ytrain)
        models.append(model)
    
    return models , xtrain.columns

def predict_gbt(
    model ,
    X ,
    train_vars ,
    date ,
    pred_dict ,
    l ,
    target_variable = target_variable,
    ) :
    X = X[train_vars]
    
    preds = []
    for mod in model:
        prediction = mod.predict(X)[0]
        preds.append(prediction)
    
    pred_dict[l].append(np.nanmean(preds))
    return pred_dict

'''
DEFINE LSTM FUNCTIONS
'''

def fit_lstm(
    ttrain ,
    gdp_lags = 0,
    target_variable = target_variable,
    params = {
        "n_timesteps": 12,
        "fill_na_func": np.nanmean,
        "fill_ragged_edges_func": np.nanmean,
        "n_models": 100,
        "train_episodes": 100,
        "batch_size": 50,
        "decay": 0.98,
        "n_hidden": 10,
        "n_layers": 1,
        "dropout": 0.0,
        "criterion": torch.nn.MSELoss(),
        "optimizer": torch.optim.Adam,
        "optimizer_parameters": {"lr": 1e-2, "weight_decay": 0.0}
    }
    ) :
    
    train = lagged_target(ttrain, gdp_lags)
    
    model = LSTM(
        data = train,
        target_variable = target_variable ,
        **params
        )
    model.train(quiet=True)
    
    return model , train.drop(["date", target_variable], axis=1).columns

def predict_lstm(
    model ,
    X ,
    train_vars ,
    date ,
    pred_dict ,
    l ,
    gdp_lags = 0 ,
    target_variable = target_variable,
    ) :
    
    X = lagged_target(X, gdp_lags)
    #X = X[train_vars]
    
    pred = model.predict(X).loc[lambda x: x.date == date, "predictions"].values[0]
    
    pred_dict[l].append(pred)
    return pred_dict
```

### LASSO

### OLS Ridge

### ENET

### Decision Tree

### Random Forest

### Gradient Boosting Tree

### Long Term Short Memory (LSTM)